# CruxSight.ai — CST-GNN Training Notebook

**Theory of Constraints-Driven Causal GNNs for
Microservices Bottleneck Detection and Management**
Maral Alshanaa · 4th Year Graduation Project · 2026

---

## Overview
This notebook trains and evaluates the CST-GNN model on the
DeathStarBench Social Network dataset (PACE Lab, Stony Brook University).

**Sections:**
1. Installation & Setup
2. Model Architecture (CST-GNN)
3. Dataset Loading (from Drive cache)
4. Loss Function & Evaluator
5. Training (Run 4 — final configuration)
6. Evaluation on Compose Workflow (N=30)
7. Generalization Study — Home Workflow (N=7)
8. Save Final Results

**Dataset cache required:**
Upload the 4 cache files to your Drive at:
`/content/drive/MyDrive/bottleneck_project/cst_gnn/dataset_cache/`
- train.pt, val.pt, test.pt, graphs.pt (~27MB total)

**Pretrained checkpoint:**
`results/checkpoints/final_model_v1.pt` (available in the GitHub repo)

# CruxSight.ai
### Causal Bottleneck Prediction for Microservices
#### Using Theory of Constraints + Graph Neural Networks

[badge: Python] [badge: PyTorch] [badge: Status: Research]

## The Problem
...one paragraph, non-technical

## The Approach (one diagram)
...CST-GNN architecture figure

## Key Results
| Metric          | Value  |
|-----------------|--------|
| Detection AUC   | 0.869  |
| Pattern Accuracy| 88.9%  |
| Zero-shot AUC   | 0.544  |
| Fine-tuned AUC  | 0.906  |

## Dataset
DeathStarBench (PACE Lab, Stony Brook)...

## Quick Start
...3 commands to run inference

## Paper
Coming soon — [CruxSight.ai](https://cruxsight.ai)

## 1. Setup & Installation
Installs PyTorch Geometric and verifies GATConv works in this environment.

In [ ]:
# ============================================================
#  1. Setup & Installation
# ============================================================

import subprocess, sys

# Install only torch_geometric core — GATConv works without
# torch-scatter/torch-sparse in PyG >= 2.5 via native PyTorch ops
result = subprocess.run(
    [sys.executable, '-m', 'pip', 'install',
     'torch_geometric', '--quiet'],
    capture_output=True, text=True
)
print(result.stdout[-500:] if result.stdout else "")
if result.returncode != 0:
    print("STDERR:", result.stderr[-1000:])

# Verify
try:
    from torch_geometric.nn import GATConv
    from torch_geometric.utils import dense_to_sparse
    import torch_geometric
    print(f"\n✓ PyTorch Geometric {torch_geometric.__version__} ready")

    # Quick functional test of GATConv
    import torch
    test_x  = torch.randn(10, 16)
    test_ei = torch.randint(0, 10, (2, 20))
    layer   = GATConv(16, 8, heads=2)
    out     = layer(test_x, test_ei)
    print(f"✓ GATConv test passed — output shape: {out.shape}")

except Exception as e:
    print(f"\n✗ Still failing: {e}")
    print("\nFallback: restart runtime (Runtime → Restart session)")
    print("then run Cell 1 again, then this cell, skipping the")
    print("torch-scatter/torch-sparse line entirely.")

## 2. Environment Setup & Configuration
Mounts Google Drive, restores the Config dataclasses, and loads the
ToC priors (patterns A–G, critical path, capacity arrays).

In [ ]:
# ============================================================
# 2: from google.colab import drive; drive.mount
# ============================================================

import os, json, warnings
import numpy as np
import torch
warnings.filterwarnings('ignore')

from google.colab import drive
drive.mount('/content/drive')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

DRIVE_BASE  = '/content/drive/MyDrive/bottleneck_project'
RESULTS_DIR = f'{DRIVE_BASE}/results'
MODEL_DIR   = f'{DRIVE_BASE}/cst_gnn'
CACHE_DIR   = f'{MODEL_DIR}/dataset_cache'

# ── Restore Config classes ────────────────────────────────
from dataclasses import dataclass, field
from typing import List, Dict

@dataclass
class DataConfig:
    window_steps:  int   = 12
    horizon_steps: int   = 6
    step_sec:      int   = 10
    min_samples:   int   = 500
    n_features:    int   = 7
    train_ratio:   float = 0.70
    val_ratio:     float = 0.15
    random_seed:   int   = 42
    batch_size:    int   = 64
    num_workers:   int   = 2

@dataclass
class ModelConfig:
    gat_in_feats:   int   = 7
    gat_hidden:     int   = 64
    gat_heads:      int   = 4
    gat_layers:     int   = 2
    gat_dropout:    float = 0.1
    toc_lambda:     float = 2.0
    toc_gamma:      float = 0.5
    tft_hidden:     int   = 128
    tft_heads:      int   = 4
    lstm_layers:    int   = 2
    tft_dropout:    float = 0.1
    causal_hidden:  int   = 64
    dag_reg:        float = 1.0
    n_patterns:     int   = 8

@dataclass
class TrainingConfig:
    epochs:         int   = 60
    lr:             float = 1e-3
    weight_decay:   float = 1e-4
    grad_clip:      float = 1.0
    patience:       int   = 10
    lambda_pattern: float = 0.5
    lambda_ttb:     float = 0.3
    lambda_causal:  float = 0.2
    lambda_sub:     float = 0.1
    fn_weight:      float = 5.0
    fp_weight:      float = 1.0
    constraint_mult:float = 3.0

@dataclass
class Config:
    data:     DataConfig     = field(default_factory=DataConfig)
    model:    ModelConfig    = field(default_factory=ModelConfig)
    training: TrainingConfig = field(default_factory=TrainingConfig)
    device:   str            = 'cuda'
    results_dir: str         = RESULTS_DIR
    model_dir:   str         = MODEL_DIR

    @classmethod
    def load(cls, path: str) -> 'Config':
        import yaml
        with open(path) as f:
            d = yaml.safe_load(f)
        cfg          = cls()
        cfg.data     = DataConfig(**d['data'])
        cfg.model    = ModelConfig(**d['model'])
        cfg.training = TrainingConfig(**d['training'])
        cfg.device   = d.get('device', 'cuda')
        return cfg

# Load the saved config from Drive
cfg = Config.load(f'{MODEL_DIR}/config.yaml')
print(f"✓ Config restored — window={cfg.data.window_steps}, "
      f"patterns={cfg.model.n_patterns}")

# ── Restore TOC priors (load cached capacity arrays — fast) ──
class TOCPriorLoader:
    PATTERNS = {
        'A': frozenset([4,5,7,8,11,12,13,14,18,19,20,21,26,27,28]),
        'B': frozenset([4,5,13,14,20,21,26,27,28]),
        'C': frozenset([0,1,2,22]),
        'D': frozenset([0,1,2,13,14,20,21,22,26,27,28]),
        'E': frozenset([0,1,2,4,5,7,8,11,12,13,14,18,19,20,21,22,26,27,28]),
        'F': frozenset([0,1,2,4,5,7,8,11,12,18,19]),
        'G': frozenset([3,4]),
        'none': frozenset(),
    }
    PATTERN_TO_IDX = {p: i for i, p in
                      enumerate(['A','B','C','D','E','F','G','none'])}
    IDX_TO_PATTERN = {v: k for k, v in PATTERN_TO_IDX.items()}
    CRITICAL_PATH = frozenset([0,1,2,4,5,7,8,11,12,13,14,
                                18,19,20,21,26,27,28])
    STORAGE_CORE  = frozenset([13,14,20,21,26,27,28])
    SILENT_NODES  = frozenset([3,6,9,10,15,16,17,23,24,25,29])

    def __init__(self, capacity_compose, capacity_home):
        self.capacity_compose = capacity_compose
        self.capacity_home    = capacity_home

    def identify_pattern(self, flagged: set) -> str:
        flagged_fs = frozenset(int(n) for n in flagged)
        for name, nodes in self.PATTERNS.items():
            if flagged_fs == nodes:
                return name
        best, best_iou = 'none', 0.0
        for name, nodes in self.PATTERNS.items():
            if name == 'none' or not nodes:
                continue
            iou = (len(flagged_fs & nodes) /
                   max(len(flagged_fs | nodes), 1))
            if iou > best_iou:
                best, best_iou = name, iou
        return best if best_iou > 0.7 else 'none'

    def get_tensor(self, n_nodes: int = 30) -> torch.Tensor:
        cap = (self.capacity_compose if n_nodes == 30
               else self.capacity_home)
        return torch.tensor(cap, dtype=torch.float32)


capacity_compose = np.load(f'{MODEL_DIR}/capacity_compose.npy')
capacity_home    = np.load(f'{MODEL_DIR}/capacity_home.npy')
toc = TOCPriorLoader(capacity_compose, capacity_home)

print(f"✓ TOC priors restored — "
      f"compose cap max={capacity_compose.max():.3f}, "
      f"home cap max={capacity_home.max():.3f}")
print(f"\nReady — now run Cell 6 (PyG import test + model definitions)")

## 3. Model Architecture — Spatial, Temporal & Causal Layers
Defines the three core CST-GNN components:
- `TOCGATLayer` / `SpatialEncoder` — capacity-biased graph attention
- `TemporalEncoder` — TFT-style LSTM + attention over the 12-step window
- `CausalInferenceLayer` — NOTEARS-inspired DAG learning (Root Cause Score)

Includes `GraphBuilder` (compose N=30 / home N=7 topologies) and a
smoke test confirming both graph sizes run correctly.

In [ ]:
# ============================================================
# 3: class TOCGATLayer(nn.Module)
# ============================================================

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GATConv
from torch_geometric.utils import dense_to_sparse
from typing import Tuple, Dict


# ═══════════════════════════════════════════════════════════
# LAYER 1: TOC-Informed Graph Attention Network
# ═══════════════════════════════════════════════════════════

class TOCGATLayer(nn.Module):
    """
    GAT layer where node features are pre-scaled by TOC capacity
    before attention is computed.

    Note: out_channels here is PER-HEAD dimension (hidden // heads).
    The layer's actual output dimension is out_channels * heads = hidden,
    so proj/norm must operate on that full dimension to keep stacked
    layers' input/output shapes consistent.
    """

    def __init__(self,
                 in_channels:  int,
                 out_channels: int,   # per-head dim = hidden // heads
                 heads:        int   = 4,
                 toc_lambda:   float = 2.0,
                 dropout:      float = 0.1):
        super().__init__()
        self.toc_lambda = toc_lambda
        self.toc_scale  = nn.Parameter(torch.tensor(1.0))

        self.gat = GATConv(in_channels, out_channels,
                           heads=heads, dropout=dropout,
                           add_self_loops=True)

        full_dim  = out_channels * heads   # = hidden
        self.proj = nn.Linear(full_dim, full_dim)
        self.norm = nn.LayerNorm(full_dim)

    def forward(self,
                x:            torch.Tensor,
                edge_index:   torch.Tensor,
                toc_capacity: torch.Tensor
                ) -> torch.Tensor:
        capacity_weight = (1.0 + self.toc_lambda *
                           self.toc_scale * toc_capacity)
        x_toc = x * capacity_weight.unsqueeze(-1)

        h = self.gat(x_toc, edge_index)   # (*, out_channels*heads)
        h = self.proj(h)                  # (*, hidden) — same dim
        h = self.norm(h)
        return F.elu(h)


class SpatialEncoder(nn.Module):
    """
    Stacks TOC-GAT layers. Processes ALL timesteps of ALL batch
    items in ONE forward pass by flattening (B, T, N) into a
    single block-diagonal graph of size B*T*N nodes.
    """

    def __init__(self,
                 in_feats: int,
                 hidden:   int,
                 n_layers: int = 2,
                 heads:    int = 4):
        super().__init__()
        self.layers = nn.ModuleList()
        self.layers.append(TOCGATLayer(in_feats, hidden // heads, heads))
        for _ in range(n_layers - 1):
            self.layers.append(TOCGATLayer(hidden, hidden // heads, heads))
        self.out_dim = hidden

    @staticmethod
    def build_batched_edge_index(edge_index: torch.Tensor,
                                  n_nodes:    int,
                                  n_graphs:   int) -> torch.Tensor:
        """
        Replicates a single graph's edge_index n_graphs times,
        offsetting node indices for each copy. This is the
        standard PyG technique for batching identical-topology
        graphs without padding or looping.

        Args:
            edge_index: (2, E) edges for ONE graph
            n_nodes:    N — nodes per graph
            n_graphs:   B*T — total number of graph copies needed

        Returns:
            (2, E * n_graphs) batched edge_index
        """
        E = edge_index.shape[1]
        device = edge_index.device
        offsets = (torch.arange(n_graphs, device=device)
                   .repeat_interleave(E) * n_nodes)
        repeated = edge_index.repeat(1, n_graphs)
        return repeated + offsets.unsqueeze(0)

    def forward(self,
                x_seq:        torch.Tensor,  # (B, T, N, F)
                edge_index:   torch.Tensor,  # (2, E) single graph
                toc_capacity: torch.Tensor   # (N,)
                ) -> torch.Tensor:
        B, T, N, Fdim = x_seq.shape

        # Flatten batch & time into the graph-copy dimension
        x_flat = x_seq.reshape(B * T * N, Fdim)

        batched_ei = self.build_batched_edge_index(
            edge_index, N, B * T
        )
        cap_flat = toc_capacity.repeat(B * T)

        h = x_flat
        for layer in self.layers:
            h = layer(h, batched_ei, cap_flat)

        # Reshape back to (B, T, N, hidden)
        return h.reshape(B, T, N, self.out_dim)


# ═══════════════════════════════════════════════════════════
# LAYER 2: Temporal Fusion Transformer (simplified)
# ═══════════════════════════════════════════════════════════

class GatedResidual(nn.Module):
    """Gated Residual Network — core TFT building block."""

    def __init__(self, d_model: int, dropout: float = 0.1):
        super().__init__()
        self.fc1  = nn.Linear(d_model, d_model)
        self.fc2  = nn.Linear(d_model, d_model)
        self.gate = nn.Linear(d_model, d_model)
        self.norm = nn.LayerNorm(d_model)
        self.drop = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        h = F.elu(self.fc1(x))
        h = self.drop(self.fc2(h))
        g = torch.sigmoid(self.gate(x))
        return self.norm(x + g * h)


class VariableSelectionNetwork(nn.Module):
    """
    Learns which of the 7 input features matter most.
    Expected to learn: latency features (0-5) > TOC prior (6)
    > [no resource features present at all — by design].
    """

    def __init__(self, d_model: int, n_vars: int):
        super().__init__()
        self.n_vars   = n_vars
        self.var_grns = nn.ModuleList(
            [GatedResidual(d_model) for _ in range(n_vars)]
        )
        self.weight_net = nn.Linear(d_model * n_vars, n_vars)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (B*N, T, n_vars, d_model) — pre-split per variable
        processed = [self.var_grns[i](x[..., i, :])
                     for i in range(self.n_vars)]
        concat  = torch.cat(processed, dim=-1)       # (B*N,T,n_vars*D)
        weights = torch.softmax(
            self.weight_net(concat), dim=-1
        ).unsqueeze(-1)                              # (B*N,T,n_vars,1)
        stacked = torch.stack(processed, dim=2)      # (B*N,T,n_vars,D)
        return (stacked * weights).sum(dim=2)        # (B*N,T,D)


class TemporalEncoder(nn.Module):
    """
    LSTM + Multi-Head Attention over the 12-step window,
    applied independently per node (all nodes processed in
    parallel by flattening B and N together).
    """

    def __init__(self,
                 d_spatial:     int,
                 d_model:       int = 128,
                 n_heads:       int = 4,
                 n_lstm_layers: int = 2,
                 dropout:       float = 0.1):
        super().__init__()

        # Project each spatial feature dim separately for VSN
        self.input_proj = nn.Linear(d_spatial, d_model)
        self.feature_split = nn.Linear(d_model, d_model * 5)
        # 5 pseudo-variables derived from the spatial embedding
        self.vsn = VariableSelectionNetwork(d_model, n_vars=5)

        self.lstm = nn.LSTM(
            input_size=d_model, hidden_size=d_model,
            num_layers=n_lstm_layers, batch_first=True,
            dropout=dropout if n_lstm_layers > 1 else 0.0,
        )
        self.attn = nn.MultiheadAttention(
            embed_dim=d_model, num_heads=n_heads,
            dropout=dropout, batch_first=True,
        )
        self.grn  = GatedResidual(d_model, dropout)
        self.norm = nn.LayerNorm(d_model)
        self.out_dim = d_model

    def forward(self, h_spatial: torch.Tensor) -> torch.Tensor:
        # h_spatial: (B, T, N, D_spatial)
        B, T, N, D = h_spatial.shape

        x = h_spatial.permute(0, 2, 1, 3).reshape(B * N, T, D)
        x = self.input_proj(x)                         # (B*N,T,d_model)

        x_split = self.feature_split(x)                # (B*N,T,5*d_model)
        x_split = x_split.view(B * N, T, 5, -1)         # (B*N,T,5,d_model)
        x_vsn   = self.vsn(x_split)                     # (B*N,T,d_model)

        x_lstm, _ = self.lstm(x_vsn)
        x_attn, _ = self.attn(x_lstm, x_lstm, x_lstm)
        x_out     = self.grn(self.norm(x_attn + x_lstm))

        x_final = x_out[:, -1, :]                       # last timestep
        return x_final.reshape(B, N, -1)                # (B, N, d_model)


# ═══════════════════════════════════════════════════════════
# LAYER 3: Causal Inference (NOTEARS-inspired)
# ═══════════════════════════════════════════════════════════

class CausalInferenceLayer(nn.Module):
    """
    Learns a sparse DAG over node embeddings.
    Root Cause Score = out-degree (causal influence) × TOC capacity.
    """

    def __init__(self, d_model: int, n_nodes: int):
        super().__init__()
        self.n_nodes = n_nodes
        self.W_raw   = nn.Parameter(torch.zeros(n_nodes, n_nodes))
        self.encoder = nn.Sequential(
            nn.Linear(d_model, d_model // 2),
            nn.ELU(),
            nn.Linear(d_model // 2, n_nodes),
        )

    def acyclicity_constraint(self, W: torch.Tensor) -> torch.Tensor:
        """h(W) = tr(e^(W∘W)) - d  via truncated series (d<=30, safe)."""
        d  = W.shape[0]
        WW = W * W
        I  = torch.eye(d, device=W.device)
        M  = I + WW / d + (WW @ WW) / (2 * d * d)
        return M.trace() - d

    def forward(self,
                h_temporal:   torch.Tensor,  # (B, N, d_model)
                toc_capacity: torch.Tensor   # (N,)
                ) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        N = self.n_nodes
        W = torch.sigmoid(self.W_raw)
        W = W * (1 - torch.eye(N, device=W.device))

        causal_contrib = torch.sigmoid(self.encoder(h_temporal))  # (B,N,N)
        causal_graph   = W.unsqueeze(0) * causal_contrib          # (B,N,N)
        causal_graph_mean = causal_graph.mean(0)                  # (N,N)

        out_degree = causal_graph.sum(dim=-1)                     # (B,N)
        rcs = out_degree * toc_capacity.unsqueeze(0)              # (B,N)

        dag_penalty = self.acyclicity_constraint(causal_graph_mean)
        return causal_graph_mean, rcs, dag_penalty


# ═══════════════════════════════════════════════════════════
# LAYER 4: Prediction Heads
# ═══════════════════════════════════════════════════════════

class PredictionHeads(nn.Module):
    def __init__(self, d_model: int, n_nodes: int, n_patterns: int = 8):
        super().__init__()
        self.pool = nn.Linear(d_model, d_model)

        head_in = d_model + n_nodes  # graph embedding + RCS vector

        self.head_bn = nn.Sequential(
            nn.Linear(head_in, 64), nn.ReLU(), nn.Dropout(0.1),
            nn.Linear(64, 1),
        )
        self.head_pattern = nn.Sequential(
            nn.Linear(head_in, 64), nn.ReLU(),
            nn.Linear(64, n_patterns),
        )
        self.head_ttb = nn.Sequential(
            nn.Linear(head_in, 64), nn.ReLU(),
            nn.Linear(64, 1), nn.Softplus(),
        )

    def forward(self,
                h_temporal: torch.Tensor,  # (B, N, d_model)
                rcs:        torch.Tensor   # (B, N)
                ) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        h_graph    = self.pool(h_temporal).mean(dim=1)   # (B, d_model)
        h_combined = torch.cat([h_graph, rcs], dim=-1)   # (B, d_model+N)

        return (self.head_bn(h_combined),
                self.head_pattern(h_combined),
                self.head_ttb(h_combined))


# ═══════════════════════════════════════════════════════════
# FULL MODEL
# ═══════════════════════════════════════════════════════════

class CSTGNN(nn.Module):
    """
    Causal Spatio-Temporal Graph Neural Network.

    forward() expects:
        x_seq:        (B, T, N, F)
        edge_index:   (2, E)  — single graph topology
        toc_capacity: (N,)

    Works for both compose (N=30) and home (N=7) graphs —
    n_nodes is inferred from x_seq.shape, all layers are
    dimension-agnostic except the prediction heads' RCS
    concatenation, which uses N dynamically too.
    """

    def __init__(self, cfg):
        super().__init__()
        m = cfg.model

        self.spatial = SpatialEncoder(
            in_feats=m.gat_in_feats,
            hidden=m.gat_hidden,
            n_layers=m.gat_layers,
            heads=m.gat_heads,
        )
        self.temporal = TemporalEncoder(
            d_spatial=m.gat_hidden,
            d_model=m.tft_hidden,
            n_heads=m.tft_heads,
            n_lstm_layers=m.lstm_layers,
            dropout=m.tft_dropout,
        )
        # Causal layer and heads are built lazily per n_nodes
        # since N differs between compose (30) and home (7)
        self._causal_cache: Dict[int, CausalInferenceLayer] = {}
        self._heads_cache:  Dict[int, PredictionHeads] = {}
        self.tft_hidden = m.tft_hidden
        self.n_patterns = m.n_patterns

    def _get_causal(self, n_nodes: int, device) -> CausalInferenceLayer:
        if n_nodes not in self._causal_cache:
            layer = CausalInferenceLayer(self.tft_hidden, n_nodes).to(device)
            self._causal_cache[n_nodes] = layer
            self.add_module(f'causal_{n_nodes}', layer)
        return self._causal_cache[n_nodes]

    def _get_heads(self, n_nodes: int, device) -> PredictionHeads:
        if n_nodes not in self._heads_cache:
            heads = PredictionHeads(
                self.tft_hidden, n_nodes, self.n_patterns
            ).to(device)
            self._heads_cache[n_nodes] = heads
            self.add_module(f'heads_{n_nodes}', heads)
        return self._heads_cache[n_nodes]

    def forward(self,
                x_seq:        torch.Tensor,  # (B, T, N, F)
                edge_index:   torch.Tensor,  # (2, E)
                toc_capacity: torch.Tensor   # (N,)
                ) -> Dict[str, torch.Tensor]:
        B, T, N, _ = x_seq.shape
        device = x_seq.device

        h_spatial  = self.spatial(x_seq, edge_index, toc_capacity)
        h_temporal = self.temporal(h_spatial)

        causal = self._get_causal(N, device)
        heads  = self._get_heads(N, device)

        causal_graph, rcs, dag_penalty = causal(h_temporal, toc_capacity)
        bn_logit, pattern_logit, ttb   = heads(h_temporal, rcs)

        return {
            'bn_logit':      bn_logit,
            'pattern_logit': pattern_logit,
            'ttb':           ttb,
            'causal_graph':  causal_graph,
            'rcs':           rcs,
            'dag_penalty':   dag_penalty,
        }

# ═══════════════════════════════════════════════════════════
# GRAPH TOPOLOGY — Adjacency / edge_index builder
# ═══════════════════════════════════════════════════════════

class GraphBuilder:
    """
    Builds edge_index (COO format) for the two known topologies:
    - Compose workflow: 30 nodes
    - Home workflow:    7 nodes

    Edge lists are based on the DeathStarBench social-network
    call graph structure observed in our 196-file analysis
    (entry layer → middle chain → storage core for compose;
     simple 7-node chain for home).
    """

    # Compose workflow (30 nodes) — directed service call edges
    COMPOSE_EDGES = [
        (0,1),(0,2),(0,22),
        (1,3),(1,4),(2,3),(2,4),
        (3,5),(3,6),(4,5),(4,6),
        (5,7),(5,8),(5,13),
        (6,7),(6,8),
        (7,9),(7,10),(8,11),(8,12),
        (13,14),(13,20),(14,21),
        (20,26),(21,27),(26,28),(27,28),
        (22,13),(22,14),
        (15,16),(16,17),(17,18),(18,19),(19,20),
    ]

    # Home workflow (7 nodes) — simple timeline-read chain
    HOME_EDGES = [
        (0,1),(0,2),(1,3),(2,3),(3,4),(4,5),(4,6),
    ]

    def __init__(self, toc_capacity: np.ndarray):
        self.capacity = toc_capacity

    def get_edge_list(self, n_nodes: int) -> list:
        return self.COMPOSE_EDGES if n_nodes == 30 else self.HOME_EDGES

    def get_edge_index(self, n_nodes: int) -> torch.Tensor:
        """
        Returns (2, E) edge_index for PyG, filtered to valid
        node indices for the given graph size. Adds reverse
        edges so GAT can pass messages in both directions
        (DeathStarBench RPCs have request/response flow).
        """
        edges = self.get_edge_list(n_nodes)
        valid = [(s, d) for s, d in edges
                 if s < n_nodes and d < n_nodes]

        if not valid:
            return torch.zeros(2, 0, dtype=torch.long)

        # Add reverse direction for bidirectional message passing
        all_edges = valid + [(d, s) for s, d in valid]
        return torch.tensor(all_edges, dtype=torch.long).t().contiguous()

    def build_dense(self, n_nodes: int) -> torch.Tensor:
        """Dense adjacency (N,N), weighted by destination capacity."""
        A = torch.zeros(n_nodes, n_nodes)
        for src, dst in self.get_edge_list(n_nodes):
            if src < n_nodes and dst < n_nodes:
                cap = (float(self.capacity[dst])
                       if dst < len(self.capacity) else 0.0)
                A[src, dst] = cap + 0.1
        return A

# ── Build and smoke-test the model ────────────────────────
print("Building CST-GNN...")
model = CSTGNN(cfg).to(device)

n_params = sum(p.numel() for p in model.parameters())
print(f"✓ Model built — {n_params:,} parameters "
      f"(causal/head layers added lazily per graph size)")

# Smoke test with synthetic data — compose graph (30 nodes)
B, T, N, Fdim = 4, cfg.data.window_steps, 30, cfg.data.n_features
x_dummy   = torch.randn(B, T, N, Fdim).to(device)
gb        = GraphBuilder(toc.capacity_compose)
edge_idx  = gb.get_edge_index(N).to(device)
cap_dummy = toc.get_tensor(N).to(device)

with torch.no_grad():
    out = model(x_dummy, edge_idx, cap_dummy)

print(f"\nSmoke test (compose, N=30):")
for k, v in out.items():
    shape = v.shape if hasattr(v, 'shape') else 'scalar'
    print(f"  {k:15s} → {shape}")

# Smoke test with home graph (7 nodes)
N7 = 7
x_dummy7   = torch.randn(B, T, N7, Fdim).to(device)
edge_idx7  = gb.get_edge_index(N7).to(device)
cap_dummy7 = toc.get_tensor(N7).to(device)

with torch.no_grad():
    out7 = model(x_dummy7, edge_idx7, cap_dummy7)

print(f"\nSmoke test (home, N=7):")
for k, v in out7.items():
    shape = v.shape if hasattr(v, 'shape') else 'scalar'
    print(f"  {k:15s} → {shape}")

print(f"\n✓ Model handles both graph sizes correctly")
print(f"Total parameters now: "
      f"{sum(p.numel() for p in model.parameters()):,}")

## 4. N-Agnostic Prediction Heads (Final Architecture)
Patches `PredictionHeads` to use a fixed-size RCS summary
(top-3 + mean + max + std), making the heads shareable across
the 30-node and 7-node graphs. This is the architecture used
for all subsequent training.

In [ ]:
# ============================================================
# the heads patch
# ============================================================

def rcs_summary(rcs: torch.Tensor, k: int = 3) -> torch.Tensor:
    """
    Converts a variable-length (B, N) root-cause-score vector
    into a fixed-size (B, k+3) summary, independent of N:
        [top-1, top-2, ..., top-k, mean, max, std]

    This is what makes PredictionHeads shareable across the
    30-node compose graph and the 7-node home graph.
    """
    topk = torch.topk(rcs, k=min(k, rcs.shape[-1]), dim=-1).values
    if topk.shape[-1] < k:  # pad if N < k (e.g. home graph if N<3)
        pad = torch.zeros(rcs.shape[0], k - topk.shape[-1],
                          device=rcs.device)
        topk = torch.cat([topk, pad], dim=-1)

    mean = rcs.mean(dim=-1, keepdim=True)
    mx   = rcs.max(dim=-1, keepdim=True).values
    std  = rcs.std(dim=-1, keepdim=True)

    return torch.cat([topk, mean, mx, std], dim=-1)  # (B, k+3)


class PredictionHeads(nn.Module):
    """
    N-AGNOSTIC version. Takes the pooled graph embedding plus
    a fixed-size (k+3) RCS summary — works for any N.
    """

    def __init__(self, d_model: int, n_patterns: int = 8,
                 summary_dim: int = 6):
        super().__init__()
        self.pool = nn.Linear(d_model, d_model)
        head_in = d_model + summary_dim

        self.head_bn = nn.Sequential(
            nn.Linear(head_in, 64), nn.ReLU(), nn.Dropout(0.1),
            nn.Linear(64, 1),
        )
        self.head_pattern = nn.Sequential(
            nn.Linear(head_in, 64), nn.ReLU(),
            nn.Linear(64, n_patterns),
        )
        self.head_ttb = nn.Sequential(
            nn.Linear(head_in, 64), nn.ReLU(),
            nn.Linear(64, 1), nn.Softplus(),
        )

    def forward(self, h_temporal, rcs):
        h_graph  = self.pool(h_temporal).mean(dim=1)   # (B, d_model)
        summary  = rcs_summary(rcs, k=3)                # (B, 6)
        combined = torch.cat([h_graph, summary], dim=-1)

        return (self.head_bn(combined),
                self.head_pattern(combined),
                self.head_ttb(combined))


# ── Patch CSTGNN: heads built once (shared), causal stays lazy ──

class CSTGNN(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        m = cfg.model

        self.spatial = SpatialEncoder(
            in_feats=m.gat_in_feats, hidden=m.gat_hidden,
            n_layers=m.gat_layers, heads=m.gat_heads,
        )
        self.temporal = TemporalEncoder(
            d_spatial=m.gat_hidden, d_model=m.tft_hidden,
            n_heads=m.tft_heads, n_lstm_layers=m.lstm_layers,
            dropout=m.tft_dropout,
        )
        # Heads are SHARED — built once, work for any N
        self.heads = PredictionHeads(m.tft_hidden, m.n_patterns)

        # Causal layer stays lazy (DAG size depends on N)
        self._causal_cache: Dict[int, CausalInferenceLayer] = {}
        self.tft_hidden = m.tft_hidden

    def _get_causal(self, n_nodes: int, device) -> CausalInferenceLayer:
        if n_nodes not in self._causal_cache:
            layer = CausalInferenceLayer(self.tft_hidden, n_nodes).to(device)
            self._causal_cache[n_nodes] = layer
            self.add_module(f'causal_{n_nodes}', layer)
        return self._causal_cache[n_nodes]

    def forward(self, x_seq, edge_index, toc_capacity):
        B, T, N, _ = x_seq.shape
        device = x_seq.device

        h_spatial  = self.spatial(x_seq, edge_index, toc_capacity)
        h_temporal = self.temporal(h_spatial)

        causal = self._get_causal(N, device)
        causal_graph, rcs, dag_penalty = causal(h_temporal, toc_capacity)

        bn_logit, pattern_logit, ttb = self.heads(h_temporal, rcs)

        return {
            'bn_logit': bn_logit, 'pattern_logit': pattern_logit,
            'ttb': ttb, 'causal_graph': causal_graph,
            'rcs': rcs, 'dag_penalty': dag_penalty,
        }


# ── Rebuild and re-validate ──────────────────────────────────
model = CSTGNN(cfg).to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f"✓ Model rebuilt — {n_params:,} parameters")
print(f"  (heads are now shared across graph sizes;"
      f" causal layer remains per-N)")

# Smoke test both graph sizes again
graphs = torch.load(f'{CACHE_DIR}/graphs.pt')

for N in [30, 7]:
    x_dummy = torch.randn(4, cfg.data.window_steps, N, cfg.data.n_features).to(device)
    ei      = graphs[N]['edge_index'].to(device)
    cap     = graphs[N]['toc_cap'].to(device)
    with torch.no_grad():
        out = model(x_dummy, ei, cap)
    print(f"\nN={N}: bn={out['bn_logit'].shape}, "
          f"pattern={out['pattern_logit'].shape}, "
          f"ttb={out['ttb'].shape}, rcs={out['rcs'].shape}")

print(f"\n✓ Heads now produce identical output shapes for N=30 and N=7")
print(f"  → home test set will use TRAINED heads, true generalization test")

## 5. Dataset, Loss Function & Evaluator
- `CachedWindowDataset` / `make_loader` — loads the cached
  train/val/test windows from Drive
- `TOCWeightedLoss` — asymmetric detection loss, pattern CE,
  TTB loss, causal sparsity, subordination loss
- Pattern class weights computed from the training distribution
- `TOCEvaluator` — AUC, F1, pattern accuracy, CP recall, subordination score

In [ ]:
# ============================================================
# dataset/loss/evaluator
# ============================================================

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from collections import Counter
import numpy as np


# ═══════════════════════════════════════════════════════════
# Dataset wrapper
# ═══════════════════════════════════════════════════════════

class CachedWindowDataset(Dataset):
    """Thin wrapper around the cached list of sample dicts."""

    def __init__(self, samples: list):
        self.samples = samples

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s = self.samples[idx]
        return s['x'], s['label'], s['pattern_idx'], s['ttb']


def make_loader(split: str, batch_size: int, shuffle: bool):
    samples = torch.load(f'{CACHE_DIR}/{split}.pt')
    ds = CachedWindowDataset(samples)
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle,
                      num_workers=0, pin_memory=True)


# ═══════════════════════════════════════════════════════════
# TOC-Weighted Loss
# ═══════════════════════════════════════════════════════════

class TOCWeightedLoss(nn.Module):
    """
    L = L_detection + λ_pat·L_pattern + λ_ttb·L_ttb
      + λ_cau·L_causal + λ_sub·L_subordination

    - Detection loss is asymmetric: missing a bottleneck (FN)
      costs more than a false alarm (FP), amplified further if
      the missed bottleneck's root cause lies on the critical path.
    - Pattern loss uses inverse-frequency class weights computed
      from the actual training distribution.
    - Subordination loss penalizes high root-cause scores on
      silent nodes (idle buffer capacity should not look "guilty").
    """

    def __init__(self, cfg, toc: 'TOCPriorLoader',
                 pattern_class_weights: torch.Tensor):
        super().__init__()
        t = cfg.training
        self.fn_weight       = t.fn_weight
        self.fp_weight       = t.fp_weight
        self.constraint_mult = t.constraint_mult
        self.lambda_pattern  = t.lambda_pattern
        self.lambda_ttb      = t.lambda_ttb
        self.lambda_causal   = t.lambda_causal
        self.lambda_sub      = t.lambda_sub

        node_weights = torch.ones(30)
        for n in toc.CRITICAL_PATH:
            node_weights[n] = t.constraint_mult
        self.register_buffer('node_weights', node_weights)

        silent_mask = torch.zeros(30)
        for n in toc.SILENT_NODES:
            silent_mask[n] = 1.0
        self.register_buffer('silent_mask', silent_mask)

        self.register_buffer('pattern_class_weights',
                             pattern_class_weights)

    def detection_loss(self, bn_logit, bn_label, rcs):
        bn_prob = torch.sigmoid(bn_logit.squeeze(-1))
        label   = bn_label.float()

        bce = F.binary_cross_entropy(bn_prob, label, reduction='none')

        pos_w = label * self.fn_weight
        neg_w = (1 - label) * self.fp_weight
        asym  = pos_w + neg_w

        crit = self.node_weights.unsqueeze(0)            # (1,N)
        rcs_crit = (rcs * crit).max(dim=-1).values
        rcs_crit = rcs_crit / (rcs_crit.max() + 1e-8)

        missed = label * (1 - (bn_prob > 0.5).float())
        amp = 1.0 + missed * rcs_crit

        return (bce * asym * amp).mean()

    def pattern_loss(self, pattern_logit, pattern_idx):
        return F.cross_entropy(pattern_logit, pattern_idx,
                               weight=self.pattern_class_weights)

    def ttb_loss(self, ttb_pred, ttb_true, bn_label):
        mask = bn_label.float().unsqueeze(-1)
        if mask.sum() == 0:
            return torch.tensor(0.0, device=ttb_pred.device)
        return F.huber_loss(ttb_pred * mask,
                            ttb_true.unsqueeze(-1) * mask,
                            delta=1.0)

    def subordination_loss(self, rcs):
        silent = self.silent_mask.unsqueeze(0)            # (1,N)
        rcs_silent = rcs * silent
        violation  = F.relu(rcs_silent - 0.1)
        return (violation ** 2).mean()

    def causal_loss(self, causal_graph, dag_penalty):
        sparsity = causal_graph.abs().mean()
        return sparsity + (dag_penalty ** 2)

    def forward(self, outputs, targets):
        L_det = self.detection_loss(outputs['bn_logit'],
                                    targets['label'], outputs['rcs'])
        L_pat = self.pattern_loss(outputs['pattern_logit'],
                                  targets['pattern_idx'])
        L_ttb = self.ttb_loss(outputs['ttb'],
                              targets['ttb'], targets['label'])
        L_sub = self.subordination_loss(outputs['rcs'])
        L_cau = self.causal_loss(outputs['causal_graph'],
                                 outputs['dag_penalty'])

        total = (L_det
                 + self.lambda_pattern * L_pat
                 + self.lambda_ttb     * L_ttb
                 + self.lambda_causal  * L_cau
                 + self.lambda_sub     * L_sub)

        return {'total': total, 'detection': L_det, 'pattern': L_pat,
                'ttb': L_ttb, 'causal': L_cau, 'subordination': L_sub}


# ═══════════════════════════════════════════════════════════
# Compute pattern class weights from actual training data
# ═══════════════════════════════════════════════════════════

train_samples = torch.load(f'{CACHE_DIR}/train.pt')
pattern_counts = Counter(int(s['pattern_idx']) for s in train_samples)

n_classes = cfg.model.n_patterns
counts = np.array([pattern_counts.get(i, 0) for i in range(n_classes)],
                   dtype=np.float32)
print("Training set pattern distribution:")
for i, c in enumerate(counts):
    name = toc.IDX_TO_PATTERN.get(i, '?')
    print(f"  {name} (idx {i}): {int(c)} samples")

# Inverse frequency weighting, smoothed
weights = 1.0 / (counts + 1.0)
weights = weights / weights.sum() * n_classes
pattern_class_weights = torch.tensor(weights, dtype=torch.float32)
print(f"\nClass weights: "
      f"{[round(w,2) for w in weights.tolist()]}")


# ═══════════════════════════════════════════════════════════
# Evaluator
# ═══════════════════════════════════════════════════════════

class TOCEvaluator:
    """Tracks predictions across a full epoch and computes
    standard + ToC-specific metrics."""

    def __init__(self, toc: 'TOCPriorLoader', n_nodes: int = 30):
        self.toc = toc
        self.n_nodes = n_nodes
        self.reset()

    def reset(self):
        self.labels, self.probs = [], []
        self.pattern_pred, self.pattern_true = [], []
        self.ttb_pred, self.ttb_true = [], []
        self.rcs_all = []

    def update(self, outputs, targets):
        probs = torch.sigmoid(outputs['bn_logit']).squeeze(-1)
        self.labels.extend(targets['label'].cpu().tolist())
        self.probs.extend(probs.detach().cpu().tolist())

        pat_pred = outputs['pattern_logit'].argmax(dim=-1)
        self.pattern_pred.extend(pat_pred.cpu().tolist())
        self.pattern_true.extend(targets['pattern_idx'].cpu().tolist())

        self.ttb_pred.extend(outputs['ttb'].squeeze(-1).detach().cpu().tolist())
        self.ttb_true.extend(targets['ttb'].cpu().tolist())

        self.rcs_all.extend(outputs['rcs'].detach().cpu().tolist())

    def compute(self) -> dict:
        from sklearn.metrics import roc_auc_score, f1_score

        labels = np.array(self.labels)
        probs  = np.array(self.probs)
        preds  = (probs > 0.5).astype(int)

        res = {}
        res['auc'] = (roc_auc_score(labels, probs)
                      if len(set(labels)) > 1 else 0.0)
        res['f1']  = f1_score(labels, preds, zero_division=0)
        res['precision'] = ((preds*labels).sum() / (preds.sum()+1e-8))
        res['recall']    = ((preds*labels).sum() / (labels.sum()+1e-8))

        pat_pred = np.array(self.pattern_pred)
        pat_true = np.array(self.pattern_true)
        res['pattern_accuracy'] = float((pat_pred == pat_true).mean())

        # Constraint-path recall (only meaningful for N=30 graphs)
        if self.n_nodes == 30:
            rcs_arr = np.array(self.rcs_all)
            crit = self.toc.CRITICAL_PATH
            correct, total = 0, 0
            for i, lab in enumerate(self.labels):
                if lab == 0:
                    continue
                total += 1
                top3 = set(np.argsort(rcs_arr[i])[-3:].tolist())
                if top3 & crit:
                    correct += 1
            res['cp_recall'] = correct / max(total, 1)

            # Subordination score: 1 - mean RCS on silent nodes
            silent = list(self.toc.SILENT_NODES)
            rcs_norm = rcs_arr / (rcs_arr.max(axis=1, keepdims=True)+1e-8)
            res['subordination_score'] = float(
                1.0 - rcs_norm[:, silent].mean()
            )
        else:
            res['cp_recall'] = None
            res['subordination_score'] = None

        # Early warning lead time (TP only)
        ttb_pred = np.array(self.ttb_pred)
        tp = (labels == 1) & (preds == 1)
        res['lead_time_min'] = (float(ttb_pred[tp].mean())
                                if tp.sum() > 0 else 0.0)

        return res


print(f"\n✓ Loss function, dataset wrapper, evaluator ready")
print(f"  Pattern class weights computed from {len(train_samples)} "
      f"training samples")

## 6. Final Training Configuration (Run 4) + Root-Cause Supervision
Applies the tuned hyperparameters from the ablation study
(fn_weight=1.5, gat_hidden=32, tft_hidden=64, λ_causal=0.05,
λ_sub=0.05) and extends the loss with direct RCS supervision
(λ_rcs_sup=0.3) — supervising the causal layer using the
pattern taxonomy (A–G).

In [ ]:
# ============================================================
# 6. Final Training Configuration (Run 4) + Root-Cause Supervision
# ============================================================

# Tuned hyperparameters from the ablation study (Run 4 — best result)
cfg.training.fn_weight      = 1.5    # was 5.0 — fixes probability saturation
cfg.training.fp_weight       = 1.0
cfg.training.lambda_causal   = 0.05  # was 0.2
cfg.training.lambda_sub      = 0.05  # was 0.1
cfg.training.lambda_rcs_sup  = 0.3   # NEW — direct root-cause supervision

cfg.model.gat_hidden   = 32          # was 64
cfg.model.tft_hidden   = 64          # was 128
cfg.model.gat_dropout  = 0.2
cfg.model.tft_dropout  = 0.2
cfg.training.weight_decay = 1e-3     # was 1e-4

cfg.training.lr       = 5e-4         # was 1e-3
cfg.training.patience = 12           # was 10
cfg.training.epochs   = 60


# ── Extend TOCWeightedLoss with direct RCS supervision ──────────
class TOCWeightedLoss(TOCWeightedLoss):  # extend, don't replace
    def __init__(self, cfg, toc, pattern_class_weights):
        super().__init__(cfg, toc, pattern_class_weights)
        self.toc_ref = toc
        self.lambda_rcs_sup = cfg.training.lambda_rcs_sup

    def rcs_supervision_loss(self, rcs, pattern_idx, label, margin=0.1):
        """For positive samples: RCS on the pattern's flagged nodes
        should exceed RCS on non-flagged nodes by `margin`."""
        n_nodes = rcs.shape[-1]
        losses = []
        for i in range(rcs.shape[0]):
            if label[i] == 0:
                continue
            pat_name = self.toc_ref.IDX_TO_PATTERN.get(int(pattern_idx[i]), 'none')
            flagged = [n for n in self.toc_ref.PATTERNS.get(pat_name, frozenset())
                       if n < n_nodes]
            if not flagged or len(flagged) >= n_nodes:
                continue
            unflagged = [n for n in range(n_nodes) if n not in flagged]
            rcs_f = rcs[i, flagged].mean()
            rcs_u = rcs[i, unflagged].mean()
            losses.append(F.relu(margin - (rcs_f - rcs_u)))
        if not losses:
            return torch.tensor(0.0, device=rcs.device)
        return torch.stack(losses).mean()

    def forward(self, outputs, targets):
        base = super().forward(outputs, targets)
        L_rcs = self.rcs_supervision_loss(
            outputs['rcs'], targets['pattern_idx'], targets['label'])
        base['total'] = base['total'] + self.lambda_rcs_sup * L_rcs
        base['rcs_supervision'] = L_rcs
        return base


# ── Rebuild model + loss with the final configuration ────────────
model   = CSTGNN(cfg).to(device)
loss_fn = TOCWeightedLoss(cfg, toc, pattern_class_weights).to(device)

n_params = sum(p.numel() for p in model.parameters())
print(f"✓ Final configuration ready — {n_params:,} parameters")
print(f"  fn_weight={cfg.training.fn_weight}  "
      f"lambda_causal={cfg.training.lambda_causal}  "
      f"lambda_sub={cfg.training.lambda_sub}  "
      f"lambda_rcs_sup={cfg.training.lambda_rcs_sup}")
print(f"  gat_hidden={cfg.model.gat_hidden}  "
      f"tft_hidden={cfg.model.tft_hidden}  lr={cfg.training.lr}")

## 7. Training Loop (Run 4 — Final Configuration)
Trains for up to 60 epochs with early stopping (patience=12).
Best checkpoint (by val AUC) is saved to
`checkpoints/best_model.pt`. Expected result: Val AUC ≈ 0.869,
Pattern Accuracy ≈ 88.9%, CP Recall consistently above the
0.944 random baseline.

In [ ]:
# ============================================================
# Training Loop
# ============================================================

import time

RUN_ID = "run4_final"  # change this each time you retrain

# ── Build loss, optimizer, scheduler ──────────────────────────
loss_fn = TOCWeightedLoss(cfg, toc, pattern_class_weights).to(device)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=cfg.training.lr,
    weight_decay=cfg.training.weight_decay,
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=cfg.training.epochs
)

# ── Data loaders ──────────────────────────────────────────────
train_loader = make_loader('train', cfg.data.batch_size, shuffle=True)
val_loader   = make_loader('val',   cfg.data.batch_size, shuffle=False)

graphs = torch.load(f'{CACHE_DIR}/graphs.pt')
edge_index_30 = graphs[30]['edge_index'].to(device)
toc_cap_30    = graphs[30]['toc_cap'].to(device)

print(f"Train batches: {len(train_loader)} | "
      f"Val batches: {len(val_loader)}")
print(f"Training for up to {cfg.training.epochs} epochs "
      f"(early stop patience={cfg.training.patience})\n")


# ── Train / eval functions ─────────────────────────────────────

def run_epoch(loader, train: bool):
    model.train() if train else model.eval()
    totals = {'total':0,'detection':0,'pattern':0,
              'ttb':0,'causal':0,'subordination':0}
    evaluator = TOCEvaluator(toc, n_nodes=30)

    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for x, label, pattern_idx, ttb in loader:
            x           = x.to(device)
            label       = label.to(device)
            pattern_idx = pattern_idx.to(device)
            ttb         = ttb.to(device)

            outputs = model(x, edge_index_30, toc_cap_30)
            targets = {'label': label, 'pattern_idx': pattern_idx,
                      'ttb': ttb}
            losses  = loss_fn(outputs, targets)

            if train:
                optimizer.zero_grad()
                losses['total'].backward()
                torch.nn.utils.clip_grad_norm_(
                    model.parameters(), cfg.training.grad_clip
                )
                optimizer.step()

            for k in totals:
                totals[k] += losses[k].item()
            evaluator.update(outputs, targets)

    n = len(loader)
    avg_losses = {k: v/n for k, v in totals.items()}
    metrics = evaluator.compute()
    return avg_losses, metrics


# ── Training loop with early stopping ───────────────────────────
best_val_auc = 0.0
best_epoch   = 0
patience_ctr = 0
history = []

print(f"{'Ep':>3} | {'TrainLoss':>9} | {'ValLoss':>8} | "
      f"{'ValAUC':>6} | {'ValF1':>5} | {'PatAcc':>6} | "
      f"{'CPRecall':>8} | {'Sub':>5} | {'Lead':>5} | LR")
print("-" * 90)

for epoch in range(1, cfg.training.epochs + 1):
    t0 = time.time()

    train_losses, train_metrics = run_epoch(train_loader, train=True)
    val_losses,   val_metrics   = run_epoch(val_loader,   train=False)

    scheduler.step()
    current_lr = scheduler.get_last_lr()[0]

    history.append({
        'epoch': epoch, 'train_loss': train_losses, 'val_loss': val_losses,
        'train_metrics': train_metrics, 'val_metrics': val_metrics,
    })

    print(f"{epoch:3d} | {train_losses['total']:9.4f} | "
          f"{val_losses['total']:8.4f} | "
          f"{val_metrics['auc']:6.4f} | {val_metrics['f1']:5.3f} | "
          f"{val_metrics['pattern_accuracy']:6.4f} | "
          f"{val_metrics['cp_recall']:8.4f} | "
          f"{val_metrics['subordination_score']:5.3f} | "
          f"{val_metrics['lead_time_min']:5.2f} | "
          f"{current_lr:.2e}  ({time.time()-t0:.1f}s)")

    # ── Checkpoint on best val AUC ────────────────────────────
    if val_metrics['auc'] > best_val_auc:
        best_val_auc = val_metrics['auc']
        best_epoch   = epoch
        patience_ctr = 0
        torch.save({
            'epoch': epoch,
            'model_state': model.state_dict(),
            'optimizer_state': optimizer.state_dict(),
            'val_metrics': val_metrics,
            'config': cfg,
        }, f'{MODEL_DIR}/checkpoints/best_model.pt')
    else:
        patience_ctr += 1
        if patience_ctr >= cfg.training.patience:
            print(f"\n⚠ Early stopping at epoch {epoch} "
                  f"(no improvement for {cfg.training.patience} epochs)")
            break

print(f"\n{'='*55}")
print(f"Training complete")
print(f"  Best epoch: {best_epoch}")
print(f"  Best val AUC: {best_val_auc:.4f}")
print(f"{'='*55}")

# Save training history
import json
with open(f'{MODEL_DIR}/logs/{RUN_ID}_history.json', 'w') as f:
    json.dump({'config': {
        'fn_weight': cfg.training.fn_weight,
        'lambda_causal': cfg.training.lambda_causal,
        'lambda_sub': cfg.training.lambda_sub,
        'lambda_rcs_sup': getattr(cfg.training, 'lambda_rcs_sup', None),
        'gat_hidden': cfg.model.gat_hidden,
        'tft_hidden': cfg.model.tft_hidden,
    }, 'history': history}, f, indent=2, default=str)

In [ ]:
# ============================================================
# 7-setup. Restore data loaders + trigger causal_30 creation
# (only needed if Cell 7's training loop was skipped)
# ============================================================

val_loader = make_loader('val', cfg.data.batch_size, shuffle=False)

graphs = torch.load(f'{CACHE_DIR}/graphs.pt')
edge_index_30 = graphs[30]['edge_index'].to(device)
toc_cap_30    = graphs[30]['toc_cap'].to(device)

# Dummy forward pass to lazily instantiate causal_30
with torch.no_grad():
    dummy = torch.randn(1, cfg.data.window_steps, 30,
                        cfg.data.n_features).to(device)
    _ = model(dummy, edge_index_30, toc_cap_30)

print("✓ val_loader, graphs, edge_index_30, toc_cap_30 restored")
print("✓ causal_30 instantiated — checkpoint can now load")

## 8. Evaluation — Best Checkpoint on Compose Validation Set
Loads `best_model.pt` and reports probability calibration and
CP Recall vs the random baseline (0.944).

In [ ]:
# ============================================================
# Diagnostics on best checkpoint
# ============================================================

import numpy as np

ckpt = torch.load(f'{MODEL_DIR}/checkpoints/best_model.pt',
                  weights_only=False)
model.load_state_dict(ckpt['model_state'])
model.eval()

evaluator = TOCEvaluator(toc, n_nodes=30)
all_probs, all_labels, all_rcs = [], [], []

with torch.no_grad():
    for x, label, pattern_idx, ttb in val_loader:
        x = x.to(device)
        outputs = model(x, edge_index_30, toc_cap_30)
        probs = torch.sigmoid(outputs['bn_logit']).squeeze(-1)
        all_probs.extend(probs.cpu().tolist())
        all_labels.extend(label.tolist())
        all_rcs.extend(outputs['rcs'].cpu().tolist())

probs  = np.array(all_probs)
labels = np.array(all_labels)
rcs    = np.array(all_rcs)

print("Probability distribution:")
print(f"  min={probs.min():.3f}  max={probs.max():.3f}  "
      f"mean={probs.mean():.3f}")
print(f"  % above 0.5: {(probs > 0.5).mean()*100:.1f}%")
print(f"  Val positive rate: {labels.mean()*100:.1f}%")

# Random baseline CPRecall for comparison
crit = list(toc.CRITICAL_PATH)
rng = np.random.default_rng(42)
random_hits = 0
for _ in range(len(labels)):
    rand_rcs = rng.random(30)
    top3 = set(np.argsort(rand_rcs)[-3:])
    if top3 & set(crit):
        random_hits += 1
print(f"\nRandom-baseline CPRecall: {random_hits/len(labels):.4f}")

# Actual CPRecall on positives only
bn_idx = np.where(labels == 1)[0]
model_hits = sum(1 for i in bn_idx
                 if set(np.argsort(rcs[i])[-3:]) & set(crit))
print(f"Model CPRecall (positives only): "
      f"{model_hits/len(bn_idx):.4f}")

# Try alternative thresholds for F1
from sklearn.metrics import f1_score, precision_score, recall_score
print(f"\nThreshold sweep:")
for thresh in [0.5, 0.6, 0.7, 0.8, 0.9]:
    preds = (probs > thresh).astype(int)
    f1 = f1_score(labels, preds, zero_division=0)
    p  = precision_score(labels, preds, zero_division=0)
    r  = recall_score(labels, preds, zero_division=0)
    print(f"  thresh={thresh}: F1={f1:.3f}  P={p:.3f}  R={r:.3f}  "
          f"%pos_pred={preds.mean()*100:.1f}%")

## 8b. Save Final Model Checkpoint

In [ ]:
import shutil
shutil.copy(f'{MODEL_DIR}/checkpoints/best_model.pt',
            f'{MODEL_DIR}/checkpoints/final_model_v1.pt')
print("✓ Saved as final_model_v1.pt")

## 9. Generalization Test — Zero-Shot on Home Workflow (N=7)
Evaluates the compose-trained model on the structurally
different, unseen 7-node home graph (Pattern G). Expected:
AUC ≈ 0.544 (near-random) — establishing the zero-shot baseline
before fine-tuning.

In [ ]:
# ============================================================
# Final evaluation on held-out Home workflow (N=7)
# ============================================================

ckpt = torch.load(f'{MODEL_DIR}/checkpoints/final_model_v1.pt',
                  weights_only=False)
model.load_state_dict(ckpt['model_state'])
model.eval()

test_loader = make_loader('test', cfg.data.batch_size, shuffle=False)
edge_index_7 = graphs[7]['edge_index'].to(device)
toc_cap_7    = graphs[7]['toc_cap'].to(device)

all_probs, all_labels, all_pat_pred, all_pat_true, all_rcs = [], [], [], [], []

with torch.no_grad():
    for x, label, pattern_idx, ttb in test_loader:
        x = x.to(device)
        outputs = model(x, edge_index_7, toc_cap_7)
        probs = torch.sigmoid(outputs['bn_logit']).squeeze(-1)

        all_probs.extend(probs.cpu().tolist())
        all_labels.extend(label.tolist())
        all_pat_pred.extend(outputs['pattern_logit'].argmax(-1).cpu().tolist())
        all_pat_true.extend(pattern_idx.tolist())
        all_rcs.extend(outputs['rcs'].cpu().tolist())

probs  = np.array(all_probs)
labels = np.array(all_labels)
rcs    = np.array(all_rcs)

from sklearn.metrics import roc_auc_score, f1_score, precision_score, recall_score

print(f"{'='*55}")
print(f"FINAL TEST RESULTS — Home Workflow (N=7, Pattern G)")
print(f"{'='*55}")
print(f"Samples: {len(labels)} | BN rate: {labels.mean()*100:.1f}%\n")

print(f"Probability range: [{probs.min():.3f}, {probs.max():.3f}], "
      f"mean={probs.mean():.3f}")

auc = roc_auc_score(labels, probs)
print(f"\nAUC: {auc:.4f}")

for t in [0.5, 0.6, 0.7]:
    preds = (probs > t).astype(int)
    print(f"  thresh={t}: F1={f1_score(labels,preds,zero_division=0):.3f}  "
          f"P={precision_score(labels,preds,zero_division=0):.3f}  "
          f"R={recall_score(labels,preds,zero_division=0):.3f}")

# Pattern G = {3, 4} for home — check RCS targeting
print(f"\nRoot-cause targeting (Pattern G = nodes {{3,4}}):")
g_nodes = [3, 4]
pos = labels == 1
top1_hits = sum(1 for i in np.where(pos)[0]
               if int(np.argmax(rcs[i])) in g_nodes)
print(f"  Top-1 RCS hits node 3 or 4: "
      f"{top1_hits}/{pos.sum()} ({top1_hits/pos.sum()*100:.1f}%)")
print(f"  (random baseline for top-1 of 7 nodes: {2/7*100:.1f}%)")

# Pattern classification — expected to fail (G unseen in training)
pat_acc = (np.array(all_pat_pred) == np.array(all_pat_true)).mean()
print(f"\nPattern accuracy: {pat_acc:.4f}")
print(f"  (Pattern G was never in training data — this measures")
print(f"   whether the model abstains/defaults sensibly on")
print(f"   out-of-distribution graph structure, not a failure)")

## 10. Fine-Tuning on Home Workflow
Two-stage fine-tuning on a small subset (169 samples, ≈25 min
of data): (1) detection-only BCE loss recovers AUC to ≈0.90 in
8 epochs, (2) adding RCS supervision for Pattern G ({3,4}) over
16 total epochs raises root-cause top-1 accuracy from 0% to
≈48.7% (vs 28.6% random baseline).

In [ ]:
# ============================================================
# 10. Fine-Tuning on Home Workflow (N=7)
# ============================================================

from torch.utils.data import random_split
from sklearn.metrics import roc_auc_score

# ── Split home samples: fine-tune set + held-out test ────────────
home_samples = torch.load(f'{CACHE_DIR}/test.pt')
n_total    = len(home_samples)
n_finetune = int(0.2 * n_total)   # ~169 samples ≈ 25 min of production data
n_holdout  = n_total - n_finetune

finetune_set, holdout_set = random_split(
    [CachedWindowDataset(home_samples).samples[i] for i in range(n_total)],
    [n_finetune, n_holdout],
    generator=torch.Generator().manual_seed(42)
)
finetune_loader = DataLoader(CachedWindowDataset(list(finetune_set)),
                              batch_size=32, shuffle=True)
holdout_loader  = DataLoader(CachedWindowDataset(list(holdout_set)),
                              batch_size=32, shuffle=False)

print(f"Fine-tune set: {len(finetune_set)} samples")
print(f"Held-out test: {len(holdout_set)} samples\n")

# ── Load the compose-trained checkpoint as starting point ─────────
ckpt = torch.load(f'{MODEL_DIR}/checkpoints/final_model_v1.pt',
                  map_location=device, weights_only=False)
missing, unexpected = model.load_state_dict(ckpt['model_state'], strict=False)
print(f"Loaded final_model_v1.pt "
      f"(missing: {len(missing)} causal_7 keys — expected, "
      f"unexpected: {len(unexpected)})\n")

ft_optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-3)


def rcs_sup_loss_g(rcs, margin=0.1):
    """RCS on nodes {3,4} (Pattern G) should exceed RCS elsewhere."""
    flagged   = [3, 4]
    unflagged = [n for n in range(rcs.shape[-1]) if n not in flagged]
    rcs_f = rcs[:, flagged].mean(dim=-1)
    rcs_u = rcs[:, unflagged].mean(dim=-1)
    return F.relu(margin - (rcs_f - rcs_u)).mean()


def evaluate_home(loader):
    model.eval()
    probs, labels, rcs_all = [], [], []
    with torch.no_grad():
        for x, label, pattern_idx, ttb in loader:
            x = x.to(device)
            outputs = model(x, edge_index_7, toc_cap_7)
            probs.extend(torch.sigmoid(outputs['bn_logit']).squeeze(-1).cpu().tolist())
            labels.extend(label.tolist())
            rcs_all.extend(outputs['rcs'].cpu().tolist())
    probs, labels, rcs_all = np.array(probs), np.array(labels), np.array(rcs_all)
    auc = roc_auc_score(labels, probs)
    pos = labels == 1
    top1 = sum(1 for i in np.where(pos)[0] if int(np.argmax(rcs_all[i])) in [3, 4])
    return auc, top1 / pos.sum() * 100


results = {}
results['Zero-shot']            = evaluate_home(holdout_loader)

# ── Stage 1: detection-only fine-tuning (8 epochs) ─────────────────
model.train()
for _ in range(8):
    for x, label, pattern_idx, ttb in finetune_loader:
        x = x.to(device)
        outputs = model(x, edge_index_7, toc_cap_7)
        bn_prob = torch.sigmoid(outputs['bn_logit'].squeeze(-1))
        loss = F.binary_cross_entropy(bn_prob, label.to(device).float())
        ft_optimizer.zero_grad()
        loss.backward()
        ft_optimizer.step()

results['+ Fine-tune (detection)'] = evaluate_home(holdout_loader)

# ── Stage 2: + RCS supervision (8 more epochs, 16 total) ────────────
model.train()
for _ in range(8):
    for x, label, pattern_idx, ttb in finetune_loader:
        x = x.to(device)
        outputs = model(x, edge_index_7, toc_cap_7)
        bn_prob  = torch.sigmoid(outputs['bn_logit'].squeeze(-1))
        loss_bce = F.binary_cross_entropy(bn_prob, label.to(device).float())
        loss_rcs = rcs_sup_loss_g(outputs['rcs'])
        loss = loss_bce + 0.3 * loss_rcs
        ft_optimizer.zero_grad()
        loss.backward()
        ft_optimizer.step()

results['+ Fine-tune (causal, 16ep)'] = evaluate_home(holdout_loader)

# ── Summary table ───────────────────────────────────────────────────
print(f"{'Stage':<28} {'AUC':>8} {'RCS Top-1':>12}")
print("-" * 50)
for stage, (auc, top1) in results.items():
    print(f"{stage:<28} {auc:>8.4f} {top1:>11.1f}%")
print(f"\nRandom baselines: AUC=0.500, RCS Top-1=28.6%")

# ── Save final fine-tuned checkpoint ────────────────────────────────
torch.save({'model_state': model.state_dict()},
          f'{MODEL_DIR}/checkpoints/final_model_v1_finetuned_home_v2.pt')
print(f"\n✓ Saved final_model_v1_finetuned_home_v2.pt")

## 11. Final Results Summary
Saves the complete ablation table (4 training configurations)
and the 3-stage generalization study to `final_results.json`
and `ablation_table.json` for the thesis and GitHub repo.

In [ ]:
# ============================================================
# CELL 12 — Final results summary (save for thesis/GitHub)
# ============================================================

ablation = [
    {'run': 1, 'fn_weight': 5.0, 'lambda_causal': 0.20, 'lambda_sub': 0.10,
     'rcs_sup': 0.0, 'params': 779921, 'best_auc': 0.8268,
     'cprecall_range': '0.84-1.00', 'note': 'probabilities saturated'},
    {'run': 2, 'fn_weight': 1.5, 'lambda_causal': 0.05, 'lambda_sub': 0.05,
     'rcs_sup': 0.0, 'params': 205617, 'best_auc': 0.8765,
     'cprecall_range': '0.58-0.87', 'note': 'CPRecall below random baseline'},
    {'run': 3, 'fn_weight': 5.0, 'lambda_causal': 0.20, 'lambda_sub': 0.10,
     'rcs_sup': 0.3, 'params': 779921, 'best_auc': 0.8236,
     'cprecall_range': '0.92-1.00', 'note': 'val loss diverged'},
    {'run': 4, 'fn_weight': 1.5, 'lambda_causal': 0.05, 'lambda_sub': 0.05,
     'rcs_sup': 0.3, 'params': 205617, 'best_auc': 0.8687,
     'cprecall_range': '0.91-1.00', 'note': 'BEST — stable + above baseline'},
]

final_results = {
    'ablation_table': ablation,  # from Cell 9d, 4 training configs
    'generalization_study': {
        'zero_shot':          {'auc': 0.5436, 'rcs_top1': 0.000},
        'finetune_detection': {'auc': 0.8995, 'rcs_top1': 0.000,
                               'epochs': 8, 'samples': 169},
        'finetune_full':      {'auc': 0.9059, 'rcs_top1': 0.487,
                               'epochs': 16, 'samples': 169},
    },
    'random_baselines': {
        'cp_recall_compose': 0.9436,   # 30-node, top-3 of 30
        'rcs_top1_home':     0.286,    # 7-node, top-1 of 7
    },
    'best_compose_model': {
        'val_auc': 0.8687, 'pattern_acc': 0.889,
        'cp_recall_avg': '0.91-1.00', 'epoch': 37,
    },
}

with open(f'{MODEL_DIR}/logs/final_results.json', 'w') as f:
    json.dump(final_results, f, indent=2)

print("✓ Final results saved")
print("\nModeling phase complete:")
print(f"  - 4 training configurations (ablation)")
print(f"  - 1 best model (Val AUC 0.869, compose, N=30)")
print(f"  - 3-stage generalization study (home, N=7)")
print(f"\nReady for Chapter 2: Literature Review")